# V1 — ssm-phylo: full pipeline in ONE Colab VM

Runs the whole V1 flow sequentially: **setup → simulate small data → train → evaluate**. Colab gives every notebook a fresh VM, so the environment (deps, Drive mount, repo clone, env vars) does NOT persist between notebooks — this master notebook keeps everything in one session.

**Every cell is idempotent**: re-running (or resuming after a session death) never duplicates work and never errors on already-completed steps. Set `COLAB_DRIVE_SKIP=1` to run headless without Drive (used by CI/nbconvert; local tmp dirs under `/tmp/ssm_v1_colab`).

### Table of contents
1. [Section 1 — Setup](#section-1--setup)
2. [Section 2 — Simulate small data](#section-2--simulate-small-data)
3. [Section 3 — Train](#section-3--train)
4. [Section 4 — Evaluate](#section-4--evaluate)


---
## Section 1 — Setup
GPU detection, Drive mount (skippable), repo bootstrap, dependency install, opt-in weight download, and a model smoke. All env vars are set in ONE cell and read from `os.environ` everywhere else — nothing is hardcoded downstream.


In [ ]:
import os, subprocess, sys, tempfile, pathlib, shutil
import torch

def _find_repo_root(start):
    """Walk up from `start` to the dir containing scripts/colab_setup.sh."""
    d = os.path.abspath(start)
    while True:
        if os.path.isfile(os.path.join(d, "scripts", "colab_setup.sh")):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return None
        d = parent

DRIVE_SKIP = os.environ.get("COLAB_DRIVE_SKIP", "0") == "1"
# nbconvert/Colab kernels run with cwd = the notebook's directory, NOT the
# repo root — locate the repo by walking up instead of trusting cwd.
REPO_DIR = _find_repo_root(os.getcwd()) or os.path.abspath(os.getcwd())
os.chdir(REPO_DIR)
print("repo dir:", REPO_DIR, "| drive skip:", DRIVE_SKIP)

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability()
    print("gpu:", torch.cuda.get_device_name(0), "| compute capability:", cc)
    if cc[0] >= 8:
        print("PROFILE: FULL (sm_80+) -> bf16 precision + fused mamba kernels")
    else:
        print("PROFILE: DEV (T4, sm_75) -> fp16 precision + eager mamba fallback")
else:
    print("PROFILE: CPU/dev -> fp32, eager mamba (no GPU)")
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)


In [ ]:
#@title Drive mount + environment (one cell; COLAB_DRIVE_SKIP=1 -> local tmp dirs)
# Belt-and-suspenders for the /usr/bin/python3 (system-python) kernel case:
# if the ssm_phylo package is not pip-installed in that interpreter, make the
# repo src/ importable directly. PYTHONPATH is inherited by subprocess cells,
# so this also covers the `python -m ssm_phylo.*` calls below.
import sys, os
_src = os.path.join(os.path.abspath(os.getcwd()), "src")
if not os.path.isdir(_src):
    _src = os.path.join(REPO_DIR, "src")  # repo lives at /content/ssm-phylo after bootstrap
sys.path.insert(0, _src)
os.environ["PYTHONPATH"] = _src + os.pathsep + os.environ.get("PYTHONPATH", "")

# Deterministic local base so headless re-runs (CI) reuse the same dirs and
# idempotency checks (e.g. "train.parquet exists") work across processes.
if not DRIVE_SKIP:
    from google.colab import drive
    drive.mount("/content/drive")
    COLAB_DRIVE = os.environ.get("COLAB_DRIVE", "/content/drive/MyDrive/ssm-phylo")
    LOCAL_CKPT_DIR = os.environ.get("LOCAL_CKPT_DIR", "/content/ckpts")
    LOCAL_DATA_DIR = os.environ.get("LOCAL_DATA_DIR", "/content/data")
else:
    COLAB_DRIVE = os.environ.get("COLAB_DRIVE", "/tmp/ssm_v1_colab")
    LOCAL_CKPT_DIR = os.environ.get("LOCAL_CKPT_DIR", os.path.join(COLAB_DRIVE, "local_ckpts"))
    LOCAL_DATA_DIR = os.environ.get("LOCAL_DATA_DIR", os.path.join(COLAB_DRIVE, "local_data"))

os.environ.update(
    COLAB_DRIVE=COLAB_DRIVE,
    DATA_DIR=os.environ.get("DATA_DIR", f"{COLAB_DRIVE}/data"),
    CKPT_DIR=os.environ.get("CKPT_DIR", f"{COLAB_DRIVE}/checkpoints"),
    RESULTS_DIR=os.environ.get("RESULTS_DIR", f"{COLAB_DRIVE}/results"),
    PROT_MAMBA_CKPT=os.environ.get("PROT_MAMBA_CKPT", f"{COLAB_DRIVE}/weights/protmamba"),
    LOCAL_CKPT_DIR=LOCAL_CKPT_DIR,
    LOCAL_DATA_DIR=LOCAL_DATA_DIR,
)
for d in [COLAB_DRIVE, os.environ["DATA_DIR"], os.environ["CKPT_DIR"],
          os.environ["RESULTS_DIR"], f"{COLAB_DRIVE}/weights",
          os.environ["LOCAL_CKPT_DIR"], os.environ["LOCAL_DATA_DIR"]]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print("COLAB_DRIVE:", COLAB_DRIVE)
print("DATA_DIR:", os.environ["DATA_DIR"])
print("LOCAL_CKPT_DIR:", os.environ["LOCAL_CKPT_DIR"])


In [ ]:
#@title Repo bootstrap (clone once; never overwrite local edits)
REPO_URL = "https://github.com/amichae2/SSM-Phylo.git"  #@param {type:"string"}
if DRIVE_SKIP:
    print("headless mode: using the current checkout (", REPO_DIR, ")")
elif os.path.isdir("/content/ssm-phylo"):
    print("repo already at /content/ssm-phylo — skipping clone (keeps local edits)")
else:
    subprocess.run(["git", "clone", REPO_URL, "/content/ssm-phylo"], check=True)
    os.chdir("/content/ssm-phylo")
    REPO_DIR = "/content/ssm-phylo"
print("using repo at:", REPO_DIR)


In [ ]:
#@title Install dependencies (colab_setup.sh; idempotent, never fails on mamba-ssm build)
# Prepend the notebook interpreter's dir to PATH so colab_setup.sh's internal
# `python` resolves to THIS python (venv in CI, system python3 on Colab).
_env = {**os.environ,
        "PATH": os.path.dirname(sys.executable) + os.pathsep + os.environ.get("PATH", ""),
        "COLAB_DRIVE": os.environ["COLAB_DRIVE"]}
setup = subprocess.run(
    ["bash", f"{REPO_DIR}/scripts/colab_setup.sh"],
    env=_env,
    capture_output=True, text=True,
)
print(setup.stdout[-2500:])
print("setup exit:", setup.returncode)
print("ssm_phylo installed." if "ssm_phylo installed." in setup.stdout else "WARN: package install line missing from setup output")


In [ ]:
#@title Download ProtMamba weights? (degraded_protmamba mode ONLY)
download_weights = False  #@param {type:"boolean"}
wm = os.environ["PROT_MAMBA_CKPT"]
already = os.path.isdir(wm) and any(os.listdir(wm))
if already:
    print("weights already present in", wm, "— skipping download")
elif download_weights:
    dl = subprocess.run(["bash", f"{REPO_DIR}/scripts/download_weights.sh"],
                        env=os.environ, capture_output=True, text=True)
    print(dl.stdout[-1500:])
else:
    print("skipped — from_scratch needs no weights (license-clean default)")


In [ ]:
#@title Smoke: imports + tiny from_scratch model
# Belt-and-suspenders for the /usr/bin/python3 (system-python) kernel case:
# if the ssm_phylo package is not pip-installed in that interpreter, make the
# repo src/ importable directly. PYTHONPATH is inherited by subprocess cells,
# so this also covers the `python -m ssm_phylo.*` calls below.
import sys, os
_src = os.path.join(os.path.abspath(os.getcwd()), "src")
if not os.path.isdir(_src):
    _src = os.path.join(REPO_DIR, "src")  # repo lives at /content/ssm-phylo after bootstrap
sys.path.insert(0, _src)
os.environ["PYTHONPATH"] = _src + os.pathsep + os.environ.get("PYTHONPATH", "")

import ssm_phylo
from ssm_phylo.models.encoder import build_encoder
from ssm_phylo.models.head import PhyloModel
from types import SimpleNamespace

cfg = SimpleNamespace(
    d_model=32, n_layer=2, vocab_size=38,
    encoder=SimpleNamespace(kind="from_scratch", checkpoint_dir=None,
        mamba={"state_size": 4, "time_step_rank": 8, "conv_kernel": 3, "expand": 2},
        ptm_model_id="ChatterjeeLab/PTM-Mamba"),
)
model = PhyloModel(build_encoder(cfg), d_emb=16, max_dist=3.0)
tok = torch.randint(0, 38, (1, 64))
spans = torch.tensor([[[0, 32], [33, 64]]])
dm, embs = model(tok, spans, torch.ones(1, 2, dtype=torch.bool))
print("dm:", tuple(dm.shape), "embs:", tuple(embs.shape), "| ssm_phylo", ssm_phylo.__version__)
print("SETUP OK")


---
## Section 2 — Simulate small data
Runs `simulation --smoke` into LOCAL scratch, consolidates + splits, copies the parquet to `$DATA_DIR` atomically, then prints stats + a patristic-distance histogram. Skipped when `train.parquet` already exists (unless **overwrite** is checked).


In [ ]:
#@title Simulate (skip if train.parquet exists)
overwrite = False  #@param {type:"boolean"}
train_parquet = os.path.join(os.environ["DATA_DIR"], "train.parquet")
if os.path.exists(train_parquet) and not overwrite:
    print("train.parquet exists — skipping simulation (check 'overwrite' to regenerate)")
else:
    proc = subprocess.Popen(
        [sys.executable, "-m", "ssm_phylo.data.simulation", "--smoke"],
        env=os.environ, cwd=REPO_DIR, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    print("\nsimulation exit:", proc.wait())


In [ ]:
#@title Data stats + distance histogram
import matplotlib.pyplot as plt
import numpy as np
import pyarrow.parquet as pq
import dendropy

if os.path.exists(train_parquet):
    for split in ("train", "val", "test"):
        p = os.path.join(os.environ["DATA_DIR"], f"{split}.parquet")
        if os.path.exists(p):
            t = pq.read_table(p)
            tips = t["n_tips"].to_numpy()
            lens = np.array([len(s) for row in t["seqs"].to_pylist()[:200] for s in row])
            print(f"{split}: {len(t)} rows | n_tips {int(tips.min())}-{int(tips.max())} | median seq len {int(np.median(lens))}")
    t = pq.read_table(train_parquet, columns=["tree_newick"])
    dists = []
    for nwk in t["tree_newick"].to_pylist()[:50]:
        tree = dendropy.Tree.get(data=nwk, schema="newick")
        ndm = tree.node_distance_matrix()
        leaves = tree.leaf_nodes()
        for i in range(len(leaves)):
            for j in range(i + 1, len(leaves)):
                dists.append(float(ndm(leaves[i], leaves[j])))
    plt.figure(figsize=(6, 3))
    plt.hist(dists, bins=40)
    plt.xlabel("patristic distance (subs/site)")
    plt.ylabel("pairs")
    plt.title("True pairwise distances (train sample)")
    plt.show()
    print("n pairs:", len(dists))
else:
    print("no train.parquet — run the simulate cell first")


---
## Section 3 — Train
Runs `ssm_phylo.train` with `--resume latest` (session death is safe: rerunning this cell resumes from the latest checkpoint instead of restarting — pull-train-push handles Drive sync).

> **WARNING: Colab sessions can die at any moment.** Training is checkpoint-resumable: if the session dies, re-run this cell — it pulls the Drive mirror and continues from the last saved step (never double-counts steps).


In [ ]:
#@title Training configuration
config = "train_small"  #@param ["train_small", "train_l4"] {type:"raw"}
if DRIVE_SKIP:
    # Headless/CI: train the tiny toy model instead of the full config (fast, CPU-safe).
    toy_scratch = os.path.join(os.environ["LOCAL_DATA_DIR"], "toy_scratch")
    pathlib.Path(toy_scratch).mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, "-m", "ssm_phylo.train", "--toy",
           "--data-dir", toy_scratch, "--resume", "latest"]
    print("headless mode: training the TOY model (CI-fast) —", " ".join(cmd))
else:
    cmd = [sys.executable, "-m", "ssm_phylo.train",
           "--config", f"{REPO_DIR}/configs/{config}.yaml",
           "--resume", "latest",
           "--data-dir", os.environ["DATA_DIR"],
           "--ckpt-dir", os.environ["CKPT_DIR"]]
    if os.environ.get("RESULTS_DIR"):
        cmd += ["--results-dir", os.environ["RESULTS_DIR"]]
    extra = os.environ.get("SSM_PHYLO_NOTEBOOK_TRAIN_EXTRA", "")
    cmd += extra.split()
    print(" ".join(cmd))


In [ ]:
#@title Train (streaming output; re-run to RESUME, not restart)
proc = subprocess.Popen(cmd, env=os.environ, cwd=REPO_DIR, text=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
print("\ntrain exit:", rc)


In [ ]:
#@title Loss curve from metrics.csv
import csv

# Fresh-VM resume safety: pull the Drive mirror first so latest.pt/metrics
# reflect the last session (no-op headless).
if not DRIVE_SKIP and os.environ.get("CKPT_DIR") and os.environ.get("LOCAL_CKPT_DIR"):
    subprocess.run(["bash", f"{REPO_DIR}/scripts/sync_drive.sh", "pull"], env=os.environ, check=False)
candidates = [
    os.path.join(os.environ["LOCAL_DATA_DIR"], "metrics.csv"),
    os.path.join(os.environ["LOCAL_DATA_DIR"], "toy_scratch", "metrics.csv"),
]
csv_path = next((p for p in candidates if os.path.exists(p)), None)
if csv_path is None:
    print("no metrics.csv found — train at least once")
else:
    rows = [r for r in csv.DictReader(open(csv_path)) if r.get("loss")]
    steps = [int(r["step"]) for r in rows]
    plt.figure(figsize=(6, 3))
    plt.plot(steps, [float(r["loss"]) for r in rows], label="loss")
    plt.plot(steps, [float(r["mae"]) for r in rows], label="mae (raw)")
    plt.xlabel("global step"); plt.ylabel("loss")
    plt.legend(); plt.title(f"training curve — {len(rows)} logged steps ({csv_path})")
    plt.show()


---
## Section 4 — Evaluate
Runs `colab_eval.sh` (pull + evaluate latest checkpoint on `$DATA_DIR/test.parquet`), renders `results.csv` as a markdown table, and plots RF vs n_tips / RF vs seq_len (figures saved to `$RESULTS_DIR`). Skipped when `results.csv` already exists (unless **re-run** is checked).


In [ ]:
#@title Evaluate (skip if results.csv exists)
rerun_eval = False  #@param {type:"boolean"}
out_dir = os.path.join(os.environ["RESULTS_DIR"], "eval_v1")
results_csv = os.path.join(out_dir, "results.csv")
if os.path.exists(results_csv) and not rerun_eval:
    print(f"results.csv exists — skipping evaluation (check 're-run' to redo): {results_csv}")
else:
    if DRIVE_SKIP:
        import glob
        ckpts = sorted(glob.glob(os.path.join(os.environ["LOCAL_CKPT_DIR"], "*.pt")))
        if not ckpts:
            raise SystemExit("no checkpoints found — run Section 3 first")
        eval_cmd = [sys.executable, "-m", "ssm_phylo.evaluate",
                    "--checkpoint", ckpts[-1],
                    "--test-parquet", os.path.join(os.environ["DATA_DIR"], "test.parquet"),
                    "--out-dir", out_dir,
                    "--max-alignments", "100"]
        proc = subprocess.run(eval_cmd, env=os.environ, cwd=REPO_DIR, text=True)
    else:
        env = {**os.environ, "SSM_PHYLO_EVAL_EXTRA": "--max-alignments 200"}
        proc = subprocess.run(["bash", f"{REPO_DIR}/scripts/colab_eval.sh"],
                             env=env, cwd=REPO_DIR, text=True)
    print("\nevaluate exit:", proc.returncode)


In [ ]:
#@title results.csv as markdown
if os.path.exists(results_csv):
    rows = list(csv.DictReader(open(results_csv)))
    print(f"| {' | '.join(rows[0].keys())} |")
    print(f"|{'---|' * len(rows[0])}")
    for r in rows[:20]:
        print(f"| {' | '.join(r.values())} |")
    print(f"... {len(rows)} rows total")
else:
    print(f"no results.csv at {results_csv} — run the evaluate cell first")


In [ ]:
#@title RF vs n_tips and RF vs seq_len (saved to $RESULTS_DIR)
if os.path.exists(results_csv):
    rows = list(csv.DictReader(open(results_csv)))
    x1 = [int(r["n_tips"]) for r in rows]
    y = [float(r["rf_pred"]) for r in rows]
    x2 = [float(r["seq_len"]) for r in rows]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))
    ax1.scatter(x1, y, alpha=0.6)
    ax1.set_xlabel("n_tips"); ax1.set_ylabel("RF (predicted vs true)")
    ax1.set_title("RF vs n_tips")
    ax2.scatter(x2, y, alpha=0.6)
    ax2.set_xlabel("mean seq length"); ax2.set_ylabel("RF")
    ax2.set_title("RF vs seq_len")
    plt.tight_layout()
    fig_path = os.path.join(out_dir, "rf_plots.png")
    plt.savefig(fig_path, dpi=150)
    plt.show()
    print("saved:", fig_path)
else:
    print("no results.csv yet — run the evaluate cell first")
